# Maia3 → ONNX Export

Run this notebook on **Google Colab** (free T4 GPU or CPU) to convert a Maia3 checkpoint to ONNX.

**Output**: `maia3-5m.onnx` (~20 MB) — download it and place in your project's `model_cache/` directory.

In [ ]:
# Install dependencies
!pip install -q torch onnx onnxruntime huggingface_hub

# Clone Maia3 source to get model definitions
!git clone --depth 1 https://github.com/UofTCSSLab/Maia3.git /content/Maia3
import sys
sys.path.insert(0, '/content/Maia3')

import os
import torch
import chess
from maia3.model_registry import resolve_model_spec, apply_model_config, resolve_checkpoint_path
from maia3.models import MAIA3Model
from maia3.dataset import tokenize_board

In [ ]:
MODEL_NAME = "maia3-5m"  # or "maia3-23m", "maia3-79m"
DEVICE = "cpu"

# Resolve model config
from argparse import Namespace
cfg = Namespace(model=MODEL_NAME, device=DEVICE, checkpoint_path=None, checkpoint_filename=None,
                cache_dir='/content/model_cache', revision=None, local_files_only=False,
                force_download=False, hf_token=None, trust_checkpoint=False)
spec = resolve_model_spec(MODEL_NAME)
apply_model_config(cfg, spec)
cfg.model_spec = spec

# Download checkpoint (~20 MB)
ckpt_path = resolve_checkpoint_path(spec, checkpoint_filename=cfg.checkpoint_filename,
                                    cache_dir=cfg.cache_dir, local_files_only=False)
print(f"Checkpoint: {ckpt_path}")

In [ ]:
# Load model
model = MAIA3Model(cfg).to(DEVICE)
ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
renamed = {k.replace("smolgen", "gab"): v for k, v in state_dict.items()}
model.load_state_dict(renamed, strict=False)
model.eval()
print("Model loaded")

In [ ]:
# Build dummy input (standard starting position)
board = chess.Board()
tokens = tokenize_board(board)         # (8, 434)
tokens_t = torch.tensor(tokens, dtype=torch.long).unsqueeze(0)  # (1, 8, 434)
elo = torch.tensor([[1500, 1500]], dtype=torch.long)            # (1, 2)

# Export to ONNX
output_path = f"/content/{MODEL_NAME}.onnx"
with torch.no_grad():
    torch.onnx.export(
        model,
        (tokens_t, elo),
        output_path,
        input_names=["tokens", "elo"],
        output_names=["policy_logits", "value_logits"],
        dynamic_axes={
            "tokens": {0: "batch"},
            "elo": {0: "batch"},
            "policy_logits": {0: "batch"},
            "value_logits": {0: "batch"},
        },
        opset_version=17,
    )

import os
size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f"Exported: {output_path} ({size_mb:.1f} MB)")

In [ ]:
# Quick sanity check: run inference with ONNX Runtime
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession(output_path)
inputs = {
    "tokens": tokens_t.numpy(),
    "elo": elo.numpy(),
}
policy, value = session.run(None, inputs)
print(f"Policy shape: {policy.shape}, Value shape: {value.shape}")
print("ONNX model is valid and produces output")

### Download the ONNX file

Run the cell below to download:
1. Download `maia3-5m.onnx`
2. Place it in your project's `model_cache/` directory

In [ ]:
from google.colab import files
files.download(output_path)
print("Download started. Save the file as model_cache/maia3-5m.onnx in your project.")